# Redis Short-Term Conversation Memory Test Suite

Tests session memory recall, session isolation, 4-route preservation, and Redis data inspection.

In [ ]:
import sys
from pathlib import Path

# Resolve project root
cwd = Path.cwd().resolve()
if cwd.name == "tests":
    project_root = cwd.parent.parent
elif cwd.name == "backend":
    project_root = cwd.parent
else:
    project_root = cwd

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from backend.app.llm.groq_provider import GroqProvider
from backend.app.llm.llm_client import LLMClient
from backend.services.chat_service import ChatService
from backend.app.core.memory import get_conversation_history

provider = GroqProvider()
llm_client = LLMClient(provider)
chat_service = ChatService(llm_client)
print("ChatService initialized with Memory module!")

In [ ]:
# Test 1 — Session Memory Recall (session_id = test-001)
session_1 = "test-001"
ans1_turn1 = await chat_service.ask("My name is Vijay.", session_id=session_1)
print(f"[{session_1}] Turn 1 Answer:", ans1_turn1)

ans1_turn2 = await chat_service.ask("What is my name?", session_id=session_1)
print(f"[{session_1}] Turn 2 Answer:", ans1_turn2)
assert "Vijay" in ans1_turn2

In [ ]:
# Test 2 — Session Memory Isolation (session_id = test-002)
session_2 = "test-002"
ans2_turn1 = await chat_service.ask("What is my name?", session_id=session_2)
print(f"[{session_2}] Turn 1 Answer:", ans2_turn1)
assert "Vijay" not in ans2_turn1

In [ ]:
# Test 3 — 4 Routes Verification (direct, rag, web, sql)
session_test = "test-003"
r_direct = await chat_service.ask("What is Python?", session_id=session_test)
r_rag = await chat_service.ask("How many annual leave days does NexaTech provide?", session_id=session_test)
r_web = await chat_service.ask("What are the latest developments in agentic AI?", session_id=session_test)
r_sql = await chat_service.ask("What is the total revenue in the sales database?", session_id=session_test)

print("Direct Route:", r_direct[:100])
print("RAG Route:", r_rag[:100])
print("Web Route:", r_web[:100])
print("SQL Route:", r_sql[:100])

In [ ]:
# Test 4 — Redis Data Structure Inspection
history_1 = get_conversation_history("test-001")
print("Redis History for session 'test-001':", history_1)
assert len(history_1) > 0
assert history_1[0]["role"] == "user"
assert "Vijay" in history_1[0]["content"]